# Pêndulo Simples

Neste notebook, a ideia principal é entender como transformar a equação do pêndulo em um algoritmo em Python.

Roteiro:
1. definir os parâmetros físicos e numéricos;
2. escrever a EDO como um sistema de primeira ordem;
3. implementar Euler-Cromer;
4. entender a ideia do RK4 e implementa-lo;
5. observar a saída numerica e o erro de cada método;
6. no fim, ver uma animação como bonus visual.


A equação do pêndulo simples e:

`theta'' + (g/L) * sin(theta) = 0`

As variáveis principais do codigo são:
- `theta`: ângulo do pêndulo
- `omega`: velocidade angular
- `dt`: tamanho do passo de tempo
- `t`: vetor de tempos da simulação

Antes da implementacao do RK4, vale guardar a ideia central: em vez de usar apenas uma inclinacao para avancar no tempo, o método calcula quatro inclinacoes dentro do mesmo intervalo e combina essas informacoes para reduzir o erro numerico.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
try:
    from IPython.display import HTML
except Exception:
    HTML = None


# ============================================================
# BLOCO 1 - Parâmetros físicos e numéricos
# g, L: parâmetros do pêndulo
# theta0, omega0: condições iniciais
# dt: passo de tempo usado na simulação
# ============================================================
g = 9.81
L = 1.0

theta0 = np.deg2rad(20.0)
omega0 = 0.0

dt = 0.01
t_final = 10.0
t = np.arange(0.0, t_final + dt, dt)
N = len(t)


# ============================================================
# BLOCO 2 - Euler-Cromer
# A EDO do pêndulo e:
#   theta'' + (g/L) * sin(theta) = 0
# Escrevemos isso como um sistema de 1ª ordem:
#   theta' = omega
#   omega' = -(g/L) * sin(theta)
#
# No Euler-Cromer, atualizamos primeiro omega e depois theta.
# Esse detalhe deixa o método mais estavel do que o Euler simples
# em problemas oscilatórios.
# ============================================================
def simular_euler_cromer():
    theta = np.zeros(N)
    omega = np.zeros(N)

    theta[0] = theta0
    omega[0] = omega0

    for i in range(N - 1):
        alpha = -(g / L) * np.sin(theta[i])
        omega[i + 1] = omega[i] + alpha * dt
        theta[i + 1] = theta[i] + omega[i + 1] * dt

    return theta, omega


# ============================================================
# BLOCO 3 - Ideia do RK4 antes da implementacao
# Para uma EDO y' = f(t, y), o RK4 calcula quatro inclinacoes:
#   k1: no inicio do intervalo
#   k2: no meio, usando k1
#   k3: no meio, usando k2
#   k4: no fim, usando k3
#
# Depois disso, faz uma media ponderada dessas inclinacoes.
# O custo computacional e maior que no Euler-Cromer, mas a
# precisao costuma ser muito melhor.
# ============================================================
def derivadas(theta, omega):
    dtheta_dt = omega
    domega_dt = -(g / L) * np.sin(theta)
    return dtheta_dt, domega_dt


def simular_rk4():
    theta = np.zeros(N)
    omega = np.zeros(N)

    theta[0] = theta0
    omega[0] = omega0

    for i in range(N - 1):
        th = theta[i]
        om = omega[i]

        k1_th, k1_om = derivadas(th, om)
        k2_th, k2_om = derivadas(th + 0.5 * dt * k1_th, om + 0.5 * dt * k1_om)
        k3_th, k3_om = derivadas(th + 0.5 * dt * k2_th, om + 0.5 * dt * k2_om)
        k4_th, k4_om = derivadas(th + dt * k3_th, om + dt * k3_om)

        theta[i + 1] = th + (dt / 6.0) * (k1_th + 2 * k2_th + 2 * k3_th + k4_th)
        omega[i + 1] = om + (dt / 6.0) * (k1_om + 2 * k2_om + 2 * k3_om + k4_om)

    return theta, omega


# ============================================================
# BLOCO 4 - Solucao de referencia para medir erro
# Aqui usamos o proprio RK4 com passo de tempo bem menor.
# Essa solucao não e "exata", mas serve como referencia numerica
# mais refinada para comparar os dois métodos implementados.
# ============================================================
def solucao_referencia():
    fator_refino = 20
    dt_ref = dt / fator_refino
    t_ref = np.arange(0.0, t_final + dt_ref, dt_ref)
    n_ref = len(t_ref)

    theta_ref = np.zeros(n_ref)
    omega_ref = np.zeros(n_ref)
    theta_ref[0] = theta0
    omega_ref[0] = omega0

    for i in range(n_ref - 1):
        th = theta_ref[i]
        om = omega_ref[i]

        k1_th, k1_om = derivadas(th, om)
        k2_th, k2_om = derivadas(th + 0.5 * dt_ref * k1_th, om + 0.5 * dt_ref * k1_om)
        k3_th, k3_om = derivadas(th + 0.5 * dt_ref * k2_th, om + 0.5 * dt_ref * k2_om)
        k4_th, k4_om = derivadas(th + dt_ref * k3_th, om + dt_ref * k3_om)

        theta_ref[i + 1] = th + (dt_ref / 6.0) * (k1_th + 2 * k2_th + 2 * k3_th + k4_th)
        omega_ref[i + 1] = om + (dt_ref / 6.0) * (k1_om + 2 * k2_om + 2 * k3_om + k4_om)

    return np.interp(t, t_ref, theta_ref)


# ============================================================
# BLOCO 5 - Saídas textuais para o aluno acompanhar o resultado
# Mostramos alguns valores numéricos e o erro de cada método.
# ============================================================
def mostrar_saida_numerica(nome_metodo, theta):
    print(f"\nSaida numerica - {nome_metodo}")
    print(" i   t(s)    theta(rad)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {theta[i]:10.6f}")


def mostrar_erros(theta_euler, theta_rk4, theta_ref):
    erro_euler = np.abs(theta_euler - theta_ref)
    erro_rk4 = np.abs(theta_rk4 - theta_ref)

    print("\nErros numericos")
    print("Referencia: RK4 com passo de tempo 20 vezes menor")
    print(f"Euler-Cromer -> erro maximo: {erro_euler.max():.6e} rad | erro final: {erro_euler[-1]:.6e} rad")
    print(f"RK4          -> erro maximo: {erro_rk4.max():.6e} rad | erro final: {erro_rk4[-1]:.6e} rad")


# ============================================================
# BLOCO 6 - Funções do bonus visual
# A animação so usa os resultados ja calculados. Ela não resolve
# o problema fisico; apenas desenha a evolução temporal.
# ============================================================
def indices_animacao(total_pontos, max_frames=160):
    return np.unique(np.linspace(0, total_pontos - 1, min(max_frames, total_pontos), dtype=int))


def animar_pendulo(theta, nome_metodo):
    x = L * np.sin(theta)
    y = -L * np.cos(theta)
    frames_anim = indices_animacao(N)
    salto = max(1, frames_anim[1] - frames_anim[0]) if len(frames_anim) > 1 else 1
    intervalo_ms = max(20, int(1000 * dt * salto))
    amp = 1.1 * max(np.max(np.abs(theta)), abs(theta0))
    margem = 0.22

    fig, (ax_pendulo, ax_theta) = plt.subplots(1, 2, figsize=(12, 4.8))

    ax_pendulo.set_title(f"Pendulo simples - {nome_metodo}")
    ax_pendulo.set_xlabel("x (m)")
    ax_pendulo.set_ylabel("y (m)")
    ax_pendulo.set_xlim(-(L + margem), L + margem)
    ax_pendulo.set_ylim(-(L + margem), 0.22)
    ax_pendulo.set_aspect("equal", adjustable="box")
    ax_pendulo.grid(alpha=0.3)
    ax_pendulo.scatter([0.0], [0.0], color="black", s=28)

    haste, = ax_pendulo.plot([], [], color="tab:green", lw=2.5)
    massa, = ax_pendulo.plot([], [], "o", color="tab:green", ms=12)

    ax_theta.set_title(f"{nome_metodo} - theta(t)")
    ax_theta.set_xlabel("tempo (s)")
    ax_theta.set_ylabel("theta (rad)")
    ax_theta.set_xlim(0.0, t_final)
    ax_theta.set_ylim(-amp, amp)
    ax_theta.grid(alpha=0.3)
    linha_theta, = ax_theta.plot([], [], color="tab:green", lw=2)

    def init():
        haste.set_data([], [])
        massa.set_data([], [])
        linha_theta.set_data([], [])
        return haste, massa, linha_theta

    def update(frame):
        haste.set_data([0.0, x[frame]], [0.0, y[frame]])
        massa.set_data([x[frame]], [y[frame]])
        linha_theta.set_data(t[: frame + 1], theta[: frame + 1])
        return haste, massa, linha_theta

    ani = FuncAnimation(
        fig,
        update,
        frames=frames_anim,
        init_func=init,
        interval=intervalo_ms,
        blit=False,
        cache_frame_data=False,
    )
    fig._ani = ani
    plt.tight_layout()
    return ani


# ============================================================
# BLOCO 7 - Execucao principal
# ============================================================
theta_euler, omega_euler = simular_euler_cromer()
theta_rk4, omega_rk4 = simular_rk4()
theta_ref = solucao_referencia()

mostrar_saida_numerica("Euler-Cromer", theta_euler)
mostrar_saida_numerica("Runge-Kutta 4", theta_rk4)
mostrar_erros(theta_euler, theta_rk4, theta_ref)


## Bonus visual

A celula abaixo apenas desenha a oscilacao e o grafico `theta(t)` usando os dados ja calculados.


In [ ]:
# Bonus visual: o pêndulo oscilando ao lado do grafico theta(t)
ani = animar_pendulo(theta_rk4, "RK4")
html = HTML(ani.to_jshtml())
plt.close(ani._fig)
html
